In [ ]:
# Lab type: extend
# Course: DS203 — Feature Engineering & Pipelines
# Lesson: ColumnTransformer and the Full Preprocessing Pipeline
# Task: A working ColumnTransformer + Pipeline baseline is provided. Extend it with
#       GridSearchCV tuning, feature name recovery and importance ranking, a third
#       column branch for binary features, and a schema-robust column selector.

# Lab: Extending the Full Preprocessing Pipeline

The baseline below is the complete `ColumnTransformer` pipeline from Lesson 3.
It fits, predicts, and reaches AUC ≈ 0.87. Your job is to extend it — not rewrite it.

Each extension builds on the previous one. Run the baseline first to confirm it works,
then add the extensions in order.

**Outputs are cleared.** Run each cell to generate results.

## Setup: install dependencies and build the dataset

In [ ]:
!pip install scikit-learn pandas numpy joblib --quiet

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.inspection import permutation_importance
import joblib

np.random.seed(42)
n = 1000

# Numeric features
tenure_days      = np.random.exponential(300, n).clip(30, 1000).round(1)
last_login_days  = np.random.exponential(20, n).clip(1, 180).round(1)
support_tickets  = np.random.poisson(2, n).clip(0, 15).astype(float)
note_word_count  = np.random.poisson(40, n).clip(0, 300).astype(float)
spend_raw        = np.random.lognormal(4.5, 0.6, n)
spend_log        = np.log1p(spend_raw).round(4)
spend_per_ticket = (spend_raw / (support_tickets + 1)).round(2)
month            = np.random.randint(1, 13, n)
signup_month_sin = np.sin(2 * np.pi * month / 12).round(4)
signup_month_cos = np.cos(2 * np.pi * month / 12).round(4)

# Categorical features
region       = np.random.choice(["NA", "EMEA", "APAC", "LATAM"], n, p=[0.4, 0.3, 0.2, 0.1])
plan         = np.random.choice(["starter", "pro", "enterprise"], n, p=[0.5, 0.35, 0.15])
recency_band = np.random.choice(["1", "2", "3", "4", "5"], n)

# Binary feature (Extension 3)
is_enterprise = (plan == "enterprise").astype(int)

# Introduce missing values
for arr in [tenure_days, last_login_days, support_tickets]:
    arr[np.random.rand(n) < 0.06] = np.nan

# Target
logit = (
    -0.002 * np.where(np.isnan(tenure_days), 400, tenure_days)
    + 0.05  * np.where(np.isnan(last_login_days), 15, last_login_days)
    + 0.3   * np.where(np.isnan(support_tickets), 2, support_tickets)
    - 0.5   * (plan == "enterprise").astype(float)
    + 0.3   * (plan == "starter").astype(float)
    + np.random.normal(0, 0.3, n)
)
churn_prob = 1 / (1 + np.exp(-logit))
churned = (np.random.rand(n) < churn_prob).astype(int)

df = pd.DataFrame({
    "monthly_spend_log":  spend_log,
    "support_tickets":    support_tickets,
    "tenure_days":        tenure_days,
    "last_login_days":    last_login_days,
    "spend_per_ticket":   spend_per_ticket,
    "note_word_count":    note_word_count,
    "signup_month_sin":   signup_month_sin,
    "signup_month_cos":   signup_month_cos,
    "region":             region,
    "plan":               plan,
    "recency_band":       recency_band,
    "is_enterprise":      is_enterprise,
    "churned":            churned,
})

print(f"Dataset: {df.shape[0]} customers, churn rate {df['churned'].mean():.1%}")
print(f"Missing values: tenure_days={df['tenure_days'].isna().sum()}, last_login={df['last_login_days'].isna().sum()}, tickets={df['support_tickets'].isna().sum()}")

## Baseline: ColumnTransformer + Pipeline

Run this cell in full. It defines and fits the pipeline from Lesson 3.
All extensions below build on `full_pipeline` and `X_train` / `X_test`.

In [ ]:
numeric_cols = [
    "monthly_spend_log", "support_tickets", "tenure_days",
    "last_login_days", "spend_per_ticket", "note_word_count",
    "signup_month_sin", "signup_month_cos",
]
categorical_cols = ["region", "plan", "recency_band"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline,      numeric_cols),
    ("cat", categorical_pipeline,  categorical_cols),
])
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",        LogisticRegression(max_iter=1000, C=1.0)),
])

X = df[numeric_cols + categorical_cols]
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

full_pipeline.fit(X_train, y_train)
y_proba = full_pipeline.predict_proba(X_test)[:, 1]
print(f"Baseline AUC: {roc_auc_score(y_test, y_proba):.3f}")

## Extension 1: Tune with GridSearchCV

Search over two hyperparameters simultaneously using double-underscore notation
to reach nested parameters inside the pipeline:

| Parameter | Values to search |
|---|---|
| `model__C` | `[0.01, 0.1, 1, 10]` |
| `preprocessor__num__imputer__strategy` | `["mean", "median"]` |

Fit `GridSearchCV` on `X_train`. Print the best params and best CV AUC.
Then evaluate the best estimator on `X_test`.

In [ ]:
# your code here: define param_grid and run GridSearchCV
# param_grid = { ... }
# grid_search = GridSearchCV(full_pipeline, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
# grid_search.fit(X_train, y_train)
# print(f"Best params: {grid_search.best_params_}")
# print(f"Best CV AUC: {grid_search.best_score_:.3f}")
# best = grid_search.best_estimator_
# print(f"Test AUC (best estimator): {roc_auc_score(y_test, best.predict_proba(X_test)[:, 1]):.3f}")

pass

> **Question:** During `GridSearchCV` with `cv=5`, the imputer is re-fitted
> on each training fold. How many times does the imputer get fitted in total
> for a grid of 8 candidate combinations?
>
> *(Write your answer here.)*

## Extension 2: Feature names and permutation importance

After fitting, recover the full list of output feature names from the
fitted `ColumnTransformer` and build a ranked feature importance table.

1. Call `get_feature_names_out()` on `full_pipeline.named_steps["preprocessor"]`.
2. Run `permutation_importance` on the test set (use `n_repeats=10, random_state=42`).
3. Print the top 8 features by mean importance.

In [ ]:
# your code here: recover feature names
# feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()
# print(f"{len(feature_names)} output features")
# print(feature_names[:5])

# your code here: permutation importance
# result = permutation_importance(full_pipeline, X_test, y_test, n_repeats=10, random_state=42)
# importance_df = pd.DataFrame({
#     "feature": feature_names,
#     "importance": result.importances_mean
# }).sort_values("importance", ascending=False)
# print(importance_df.head(8).to_string(index=False))

pass

> **Question:** Feature names from the `ColumnTransformer` are prefixed with `num__`
> or `cat__`. Why do one-hot-encoded features produce more output columns than
> input columns, and how does this affect using feature importance for model explanation?
>
> *(Write your answer here.)*

## Extension 3: Add a binary passthrough branch

The column `is_enterprise` is a binary 0/1 flag that needs neither imputation
nor scaling — it should pass through the `ColumnTransformer` unchanged.

Rebuild `full_pipeline` to add a third branch:

```
("binary", "passthrough", ["is_enterprise"])
```

Update `X` to include `is_enterprise`, re-split, refit, and compare AUC to the baseline.

In [ ]:
binary_cols = ["is_enterprise"]

# your code here: rebuild preprocessor with a passthrough branch for binary_cols
# preprocessor_v2 = ColumnTransformer([
#     ("num", numeric_pipeline,    numeric_cols),
#     ("cat", categorical_pipeline, categorical_cols),
#     ("binary", "passthrough",    binary_cols),
# ])
# full_pipeline_v2 = Pipeline([
#     ("preprocessor", preprocessor_v2),
#     ("model",        LogisticRegression(max_iter=1000)),
# ])

# X_v2 = df[numeric_cols + categorical_cols + binary_cols]
# y = df["churned"]
# X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(X_v2, y, test_size=0.2, random_state=42)
# full_pipeline_v2.fit(X_train_v2, y_train_v2)
# auc_v2 = roc_auc_score(y_test_v2, full_pipeline_v2.predict_proba(X_test_v2)[:, 1])
# print(f"Baseline AUC:         {roc_auc_score(y_test, full_pipeline.predict_proba(X_test)[:, 1]):.3f}")
# print(f"AUC with is_enterprise: {auc_v2:.3f}")

pass

> **Question:** Why is `"passthrough"` the right choice for `is_enterprise`
> instead of wrapping it in a `SimpleImputer + StandardScaler` sub-pipeline?
> What would scaling a binary 0/1 column actually do to its values?
>
> *(Write your answer here.)*

## Extension 4 (Challenge): Replace hard-coded column lists with `make_column_selector`

The baseline uses explicit `numeric_cols` and `categorical_cols` lists.
If a new column is added to the input DataFrame with the wrong dtype,
it will be silently dropped.

Use `make_column_selector` to route columns by dtype instead:
- Numeric branch: `dtype_include=np.number`
- Categorical branch: `dtype_include=object`

Verify that the selector-based pipeline produces the same AUC as the baseline
on a DataFrame where `region`, `plan`, and `recency_band` are `dtype=object`.

In [ ]:
# Ensure categorical columns are object dtype so make_column_selector routes them correctly
df_typed = df[numeric_cols + categorical_cols].copy()
df_typed[categorical_cols] = df_typed[categorical_cols].astype(object)

# your code here: build a pipeline using make_column_selector
# preprocessor_sel = ColumnTransformer([
#     ("num", numeric_pipeline, make_column_selector(dtype_include=np.number)),
#     ("cat", categorical_pipeline, make_column_selector(dtype_include=object)),
# ])
# full_pipeline_sel = Pipeline([
#     ("preprocessor", preprocessor_sel),
#     ("model",        LogisticRegression(max_iter=1000)),
# ])

# X_sel = df_typed
# y = df["churned"]
# X_tr_sel, X_te_sel, y_tr_sel, y_te_sel = train_test_split(X_sel, y, test_size=0.2, random_state=42)
# full_pipeline_sel.fit(X_tr_sel, y_tr_sel)
# auc_sel = roc_auc_score(y_te_sel, full_pipeline_sel.predict_proba(X_te_sel)[:, 1])
# print(f"Selector-based AUC: {auc_sel:.3f}")

pass

> **Question:** `make_column_selector` depends on DataFrame dtype being correct.
> What production failure could occur if a categorical column arrives as
> `int64` (e.g., an encoded region ID) instead of `object`?
> How would you guard against this at the pipeline boundary?
>
> *(Write your answer here.)*

## Summary

> **Answer each question in one sentence.**

1. What does `GridSearchCV` do differently from a manual loop over hyperparameter values?
2. Why does the `ColumnTransformer` produce more output columns than input columns when categorical branches are included?
3. What happens to a new column added to the input schema when `ColumnTransformer` uses hard-coded lists vs. `make_column_selector`?
4. When is `"passthrough"` the right transformer choice for a branch, and when should you use a sub-pipeline instead?